# TEMPO: Semi-Soft Prompt Pool for Time Series Forecasting

## Experimental Evaluation Notebook

**Base Paper:** TEMPO: Prompt-based Generative Pre-trained Transformer for Time Series Forecasting (ICLR 2024)  
**Novelty:** Semi-Soft Prompt Pool -- Semantically initialized prompt pool using GPT-2 word embeddings  
**Repository:** https://github.com/sanjai-rs7/TEMPO-SemiSoftPool

---

### Prerequisites

- Google Colab with GPU enabled (Runtime > Change runtime type > GPU T4)
- Run all cells sequentially from top to bottom

---

### Experiment Summary

| Exp | Training Data | Test Data (Unseen) | Prompt Method | Purpose |
|-----|--------------|-------------------|---------------|----------|
| 1 | ETTh1 | ETTh1 | Semi-Soft | Single-dataset baseline |
| 2 | ETTh1 | ETTh1 | Semi-Soft Pool (Ours) | Single-dataset novelty |
| 3 | 6 datasets | Weather | Semi-Soft | Zero-shot baseline |
| 4 | 6 datasets | Weather | Semi-Soft Pool (Ours) | Zero-shot novelty |
| 5 | 6 datasets | ETTm1 | Semi-Soft | Zero-shot baseline (different target) |
| 6 | 6 datasets | ETTm1 | Semi-Soft Pool (Ours) | Zero-shot novelty (different target) |
| 7 | 6 datasets | Electricity | Semi-Soft | Zero-shot baseline (different domain) |
| 8 | 6 datasets | Electricity | Semi-Soft Pool (Ours) | Zero-shot novelty (different domain) |
| 9 | 6 datasets | Weather | Random Pool | Paper's pool baseline (random init) |
| 10 | 6 datasets | Weather | No Prompt | Ablation: contribution of prompts |

### Speed Modes

| Flag | GPT-2 Weights | Training Data | Epochs | Approx. Time/Experiment | Recommended For |
|------|--------------|---------------|--------|------------------------|------------------|
| `--fast` | Random | 10% | 3 | 3-5 min | Code verification |
| `--lite` | Pre-trained | 20% | 5 | 8-12 min | Quick evaluation |
| `--medium` | Pre-trained | 100% | 5 | 15-20 min | Standard evaluation |
| (none) | Pre-trained | 100% | 10 | 30-40 min | Final thesis results |

---
## 1. Environment Setup

In [ ]:
# Clone repository
!git clone https://github.com/sanjai-rs7/TEMPO-SemiSoftPool.git
%cd TEMPO-SemiSoftPool

In [ ]:
# Install dependencies
!pip install -q transformers peft huggingface_hub omegaconf einops statsmodels reformer_pytorch gdown

# Apply compatibility fixes
!sed -i 's/np.Inf/np.inf/g' tempo/utils/tools.py

# Verify GPU availability
import torch
assert torch.cuda.is_available(), 'ERROR: No GPU detected. Enable GPU in Runtime > Change runtime type.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print('Environment ready.')

---
## 2. Dataset Download

Seven benchmark datasets used in the TEMPO paper:

| Dataset | Domain | Length | Variables | Sampling |
|---------|--------|--------|-----------|----------|
| ETTh1, ETTh2 | Electricity (transformer temp.) | 17,420 | 7 | Hourly |
| ETTm1, ETTm2 | Electricity (transformer temp.) | 69,680 | 7 | 15 min |
| Weather | Meteorology (Germany) | 52,696 | 21 | 10 min |
| Electricity | Power consumption | 26,304 | 321 | Hourly |
| Traffic | Road occupancy (California) | 17,544 | 862 | Hourly |

In [ ]:
import urllib.request, os

# ETT datasets (from official ETDataset repository)
os.makedirs('dataset/ETT-small', exist_ok=True)
base = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small'
for f in ['ETTh1.csv', 'ETTh2.csv', 'ETTm1.csv', 'ETTm2.csv']:
    urllib.request.urlretrieve(f'{base}/{f}', f'dataset/ETT-small/{f}')
    print(f'Downloaded: {f}')

# Weather, Electricity, Traffic (from official Autoformer Google Drive)
import gdown
print('\nDownloading Weather, Electricity, Traffic (may take 2-5 minutes)...')
gdown.download_folder(
    'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy',
    quiet=False, output='dataset'
)

print('\nDatasets on disk:')
!find dataset -name '*.csv' | sort
print('\nDataset download complete.')

---
## 3. Prompt Pool Templates

Our novelty: 30 semantically meaningful English descriptions used to initialize the prompt pool.
Each description is tokenized and converted to GPT-2 word embeddings, providing a meaningful
starting point in the model's embedding space (as opposed to random initialization in the original paper).

In [ ]:
!python run_experiments.py --exp prompts

---
## 4. Experiments -- Priority Order

Experiments are ordered by importance. Run the first group immediately.
Subsequent groups provide additional evidence for the thesis.

---

### 4.1 PRIMARY: Single-Dataset Baseline vs Novelty (Exp 1, 2)

**Purpose:** Verify that the Semi-Soft Prompt Pool works on a single dataset before testing zero-shot.

- Experiment 1: Train on ETTh1, test on ETTh1, using standard semi-soft prompt
- Experiment 2: Train on ETTh1, test on ETTh1, using our Semi-Soft Prompt Pool

**Expected outcome:** Experiment 2 should match or slightly improve over Experiment 1.

In [ ]:
# Lite mode: pretrained GPT-2, 20% data, 5 epochs (~16-24 min total)
!python run_experiments.py --exp 1,2 --lite

In [ ]:
# To run in fast mode instead (~6-10 min total):
# !python run_experiments.py --exp 1,2 --fast

# To run in full mode for thesis-quality results (~60-80 min total):
# !python run_experiments.py --exp 1,2

---
### 4.2 CRITICAL: Zero-Shot -- Baseline vs Novelty vs Random Pool (Exp 3, 4, 9)

**Purpose:** The central thesis comparison. Three prompt strategies tested on the same zero-shot task.

- Experiment 3: Train on 6 datasets, test on Weather (unseen). Standard semi-soft prompt.
- Experiment 4: Same setup, but using our Semi-Soft Prompt Pool.
- Experiment 9: Same setup, but using the paper's random-initialized prompt pool.

**Key comparison:**
- Exp 4 vs Exp 3: Does the prompt pool help over fixed prompts?
- Exp 4 vs Exp 9: Does semantic initialization help over random initialization?

**Expected outcome:** Exp 4 (ours) should outperform both Exp 3 and Exp 9.

In [ ]:
# Lite mode (~24-36 min total)
!python run_experiments.py --exp 3,4,9 --lite

In [ ]:
# Fast mode (~9-15 min total):
# !python run_experiments.py --exp 3,4,9 --fast

# Full mode for thesis (~90-120 min total):
# !python run_experiments.py --exp 3,4,9

---
### 4.3 SUPPORTING: Zero-Shot on Different Target -- ETTm1 (Exp 5, 6)

**Purpose:** Demonstrate that the novelty generalizes to a different unseen target dataset.

- Experiment 5: Train on 6 datasets (excluding ETTm1), test on ETTm1. Semi-soft prompt.
- Experiment 6: Same setup, using our Semi-Soft Prompt Pool.

**Expected outcome:** Exp 6 should outperform Exp 5.

In [ ]:
!python run_experiments.py --exp 5,6 --lite

---
### 4.4 SUPPORTING: Zero-Shot on Different Domain -- Electricity (Exp 7, 8)

**Purpose:** Test cross-domain transfer. Electricity has 321 variables (vs 7 in ETT), testing scalability.

- Experiment 7: Train on 6 datasets (excluding Electricity), test on Electricity. Semi-soft prompt.
- Experiment 8: Same setup, using our Semi-Soft Prompt Pool.

**Expected outcome:** Exp 8 should outperform Exp 7.

In [ ]:
!python run_experiments.py --exp 7,8 --lite

---
### 4.5 ABLATION: Contribution of Prompts (Exp 10, 3)

**Purpose:** Quantify how much prompts contribute to forecasting accuracy.

- Experiment 10: Zero-shot on Weather with NO prompt at all.
- Experiment 3: Zero-shot on Weather with semi-soft prompt (already run in 4.2).

**Expected outcome:** Exp 3 should significantly outperform Exp 10, confirming that prompts are essential.

In [ ]:
!python run_experiments.py --exp 10 --lite

---
### 4.6 RUN ALL EXPERIMENTS AT ONCE

If time permits, run all 10 experiments in a single command.
Results accumulate in the same results file.

In [ ]:
# All 10 experiments in lite mode (~1.5-2 hours)
# !python run_experiments.py --exp all --lite

# All 10 experiments in full mode (~6 hours, thesis quality)
# !python run_experiments.py --exp all

---
## 5. Results and Analysis

### 5.1 Results Table

Displays MSE and MAE for all completed experiments with key comparisons.

In [ ]:
!python run_experiments.py --exp results

### 5.2 Comparison Plots

In [ ]:
!python run_experiments.py --exp plots

from IPython.display import Image, display
import os

for d in ['experiment_results_lite', 'experiment_results_fast', 'experiment_results_medium', 'experiment_results']:
    for p in ['mse_mae_comparison.png', 'baseline_vs_novelty.png']:
        path = f'{d}/plots/{p}'
        if os.path.exists(path):
            print(f'\n{path}')
            display(Image(path))

### 5.3 Raw Results (JSON)

In [ ]:
import json, os

for d in ['experiment_results_lite', 'experiment_results_fast', 'experiment_results_medium', 'experiment_results']:
    path = f'{d}/results.json'
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        print(f'\n{path} -- {len(results)} experiments completed')
        print(f'{"Name":<45} {"MSE":<10} {"MAE":<10} {"Prompt":<18} {"Time":<10}')
        print('-' * 95)
        for r in results:
            mse = f"{r['mse']:.4f}" if r.get('mse') else 'FAILED'
            mae = f"{r['mae']:.4f}" if r.get('mae') else 'FAILED'
            print(f"{r['desc']:<45} {mse:<10} {mae:<10} {r['prompt_type']:<18} {r.get('time_human','?'):<10}")

### 5.4 Training Logs

Full training output for any experiment. Replace the experiment name as needed.

In [ ]:
# View log for a specific experiment (change the name as needed)
# Available logs:
import os
for d in ['experiment_results_lite', 'experiment_results_fast', 'experiment_results_medium', 'experiment_results']:
    log_dir = f'{d}/logs'
    if os.path.exists(log_dir):
        print(f'\nLogs in {log_dir}/:')
        for f in sorted(os.listdir(log_dir)):
            print(f'  {f}')

In [ ]:
# Print last 50 lines of a specific log
# Change the path to view different experiments
!tail -50 experiment_results_lite/logs/1_single_ETTh1.log

---
## 6. Save Results to Google Drive

Colab sessions are temporary. Run this cell to permanently save all results,
checkpoints, and plots to your Google Drive before the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
save_path = '/content/drive/MyDrive/TEMPO_Results'
os.makedirs(save_path, exist_ok=True)

for d in ['experiment_results', 'experiment_results_fast', 'experiment_results_lite', 'experiment_results_medium', 'checkpoints']:
    if os.path.exists(d):
        shutil.copytree(d, f'{save_path}/{d}', dirs_exist_ok=True)
        print(f'Saved: {d}')

print(f'\nAll results saved to Google Drive: {save_path}')

---
## 7. Updating Code

If the repository is updated on GitHub, pull the latest changes.
This preserves all downloaded datasets, trained models, and experiment results.

In [ ]:
!git pull
print('Code updated. Datasets and results are preserved.')